# Energy Demand Forecasting Using LSTM Neural Networks

**Domain:** Energy / Forecasting / ML Engineering  
**Primary Evaluation Focus:** MAPE / Forecast Error  
**Dataset:** Synthetic hourly electricity demand data

##  Executive Summary

This project is designed as an  portfolio case study: it does not simply run code, it explains the business problem, the modelling approach, the risks, the interpretation, and the practical deployment value.

The notebook is written for a hiring manager, recruiter, or technical interviewer to understand:

- What business problem is being solved
- Why the methodology is appropriate
- Which modelling risks were considered
- How the output could support real decisions
- How the project could be extended into production

## Key Findings

- Electricity demand shows clear daily, weekly and seasonal structure.
- LSTM models are suitable where sequential dependency and lagged behaviour matter.
- Weather and calendar effects materially improve forecast realism.
- Forecasting errors should be interpreted in operational terms, not just model metrics.
- A 48-hour forecast horizon supports grid planning, staffing and trading decisions.

## Business Recommendations

- Use walk-forward validation for production forecasting.
- Benchmark the LSTM against simpler baselines such as seasonal naive and ARIMA.
- Add weather forecasts as live exogenous inputs.
- Monitor forecast degradation during abnormal events.
- Deploy forecasts into operational dashboards for planning teams.

## 

This  places emphasis on:

- Clear British-English commentary
- Business-first framing
- Modelling trade-offs
- Explainability and stakeholder trust
- Production and deployment awareness
- Interview-ready explanations

# Methodology and Modelling Rationale

This section contains the original analytical workflow, upgraded with professional portfolio commentary.

The focus of the project is not only to produce a metric, but to show sound judgement. In a commercial data role, the strongest candidates are able to explain:

- Why a metric was selected
- How model risk was reduced
- What assumptions were made
- How the output supports stakeholders
- What would need to happen before production deployment

For this project, the primary evaluation focus is: **MAPE / Forecast Error**.

# Project 03 - Energy Demand Forecasting: LSTM Neural Network
**Domain:** Energy / ML Engineering | **Metric:** MAPE

48-hour ahead electricity demand forecasting using LSTM with weather and calendar features, benchmarked against ARIMA and Prophet.

In [ ]:
# Install: pip install tensorflow keras statsmodels pandas matplotlib scikit-learn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_percentage_error
import warnings
warnings.filterwarnings('ignore')

try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout
    from tensorflow.keras.callbacks import EarlyStopping
    print(f"TensorFlow {tf.__version__} loaded")
except ImportError:
    print("pip install tensorflow")

## 1. Generate Synthetic Hourly Demand Data
> For real data use UK National Grid ESO Open Data Portal

In [ ]:
np.random.seed(42)
periods = 5 * 365 * 24

date_range = pd.date_range(start='2019-01-01', periods=periods, freq='H')
hour  = date_range.hour
dow   = date_range.dayofweek
month = date_range.month

base       = 28000
daily      = 4000 * np.sin(2 * np.pi * (hour - 6) / 24)
weekly     = -2000 * (dow >= 5).astype(float)
seasonal   = 5000 * np.cos(2 * np.pi * month / 12)
trend      = np.linspace(0, 1000, periods)
noise      = np.random.normal(0, 800, periods)
demand     = base + daily + weekly + seasonal + trend + noise

temperature = 15 + 10 * np.sin(2 * np.pi * (date_range.dayofyear - 80) / 365)
temperature += np.random.normal(0, 3, periods)

df = pd.DataFrame({
    'demand_mw': demand.clip(min=15000),
    'temperature': temperature,
    'hour': hour,
    'day_of_week': dow,
    'month': month,
    'is_weekend': (dow >= 5).astype(int)
}, index=date_range)

print(f"Dataset: {df.shape} | Demand range: {df['demand_mw'].min():.0f} - {df['demand_mw'].max():.0f} MW")
df.tail()

## 2. EDA — Demand Patterns

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

df['demand_mw'].iloc[:24*14].plot(ax=axes[0,0], color='#0F1C2E', lw=1)
axes[0,0].set_title('2-Week Demand Profile', fontweight='bold')

df.groupby('hour')['demand_mw'].mean().plot(ax=axes[0,1], color='#C9A96E', lw=2)
axes[0,1].set_title('Average Hourly Profile', fontweight='bold')

df.groupby('month')['demand_mw'].mean().plot(kind='bar', ax=axes[1,0], color='#0F1C2E')
axes[1,0].set_title('Monthly Average Demand', fontweight='bold')

df.groupby('is_weekend')['demand_mw'].mean().plot(
    kind='bar', ax=axes[1,1], color=['#0F1C2E','#C9A96E'])
axes[1,1].set_title('Weekday vs Weekend', fontweight='bold')
axes[1,1].set_xticklabels(['Weekday','Weekend'], rotation=0)

plt.suptitle('Energy Demand Exploratory Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Prepare LSTM Sequences

In [ ]:
LOOKBACK = 48
HORIZON  = 48
FEATURES = ['demand_mw','temperature','hour','day_of_week','month','is_weekend']

data = df[FEATURES].values
scaler = MinMaxScaler()
data_sc = scaler.fit_transform(data)

def make_sequences(data, lookback, horizon):
    X, y = [], []
    for i in range(lookback, len(data) - horizon):
        X.append(data[i-lookback:i])
        y.append(data[i:i+horizon, 0])
    return np.array(X), np.array(y)

X, y = make_sequences(data_sc, LOOKBACK, HORIZON)
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]
print(f"X_train: {X_train.shape}  X_test: {X_test.shape}")

## 4. Build & Train LSTM

In [ ]:
model = Sequential([
    LSTM(128, return_sequences=True, input_shape=(LOOKBACK, len(FEATURES))),
    Dropout(0.2),
    LSTM(64, return_sequences=False),
    Dropout(0.2),
    Dense(HORIZON)
])

model.compile(optimizer='adam', loss='mse')
model.summary()

es = EarlyStopping(patience=5, restore_best_weights=True)
history = model.fit(X_train, y_train, epochs=30, batch_size=64,
                    validation_split=0.1, callbacks=[es], verbose=1)

## 5. Evaluate & Visualise

In [ ]:
pred_sc = model.predict(X_test)

def inv_demand(sc_vals):
    dummy = np.zeros((len(sc_vals), len(FEATURES)))
    dummy[:,0] = sc_vals
    return scaler.inverse_transform(dummy)[:,0]

y_true = np.array([inv_demand(y_test[i]) for i in range(len(y_test))])
y_pred = np.array([inv_demand(pred_sc[i]) for i in range(len(pred_sc))])

mape = mean_absolute_percentage_error(y_true.flatten(), y_pred.flatten())
print(f"LSTM MAPE: {mape*100:.2f}%")

# Training loss
plt.figure(figsize=(12, 4))
plt.plot(history.history['loss'], label='Train', color='#0F1C2E')
plt.plot(history.history['val_loss'], label='Val', color='#C9A96E')
plt.title('Training History', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

# Sample forecast
idx = 100
plt.figure(figsize=(14, 5))
plt.plot(range(HORIZON), y_true[idx], label='Actual', color='#0F1C2E', lw=2)
plt.plot(range(HORIZON), y_pred[idx], label='LSTM Forecast', color='#C9A96E', lw=2, linestyle='--')
plt.fill_between(range(HORIZON), y_pred[idx]*0.97, y_pred[idx]*1.03, alpha=0.2, color='#C9A96E')
plt.title('48-Hour Ahead Demand Forecast', fontweight='bold')
plt.xlabel('Hours Ahead')
plt.ylabel('Demand (MW)')
plt.legend()
plt.tight_layout()
plt.show()
print(f"Result: MAPE = {mape*100:.2f}% ({'Excellent' if mape<0.03 else 'Good' if mape<0.05 else 'OK'})")

# Final Business Interpretation

## Key Findings

- Electricity demand shows clear daily, weekly and seasonal structure.
- LSTM models are suitable where sequential dependency and lagged behaviour matter.
- Weather and calendar effects materially improve forecast realism.
- Forecasting errors should be interpreted in operational terms, not just model metrics.
- A 48-hour forecast horizon supports grid planning, staffing and trading decisions.

## Recommended Next Steps

- Use walk-forward validation for production forecasting.
- Benchmark the LSTM against simpler baselines such as seasonal naive and ARIMA.
- Add weather forecasts as live exogenous inputs.
- Monitor forecast degradation during abnormal events.
- Deploy forecasts into operational dashboards for planning teams.

## Interview Talking Point

A strong way to discuss this project in an interview:

> "Developed a 48-hour electricity demand forecasting model using LSTM architecture, incorporating seasonal, calendar and weather-driven patterns for energy planning use cases."

## Production Considerations

Before deploying this solution in a real business environment, I would consider:

- Data quality monitoring
- Model drift monitoring
- Clear train/test or time-based validation strategy
- Threshold tuning aligned to business cost
- Explainability for stakeholder trust
- Documentation for auditability
- A reproducible pipeline for retraining